# 🛠️ Feature Engineering Ladder: Cents, Benford & Risk Multipliers (Kaggle PGS S6E11)

Welcome to the **Feature Engineering & Modeling Notebook** for **Kaggle Playground Series Season 6 Episode 11 (Predicting Credit Card Fraud)**.

### 🎯 Feature Engineering Ladder Blocks
1. **Block A: Cents & Benford Digit Signals**:
   - Exact round dollar amounts (`.00` cents) exhibit an elevated fraud rate of **2.089%** vs. **1.666%** for non-round amounts.
   - Benford's Law anomaly detection on the leading digits of `amount_usd` and `account_balance_usd`.
2. **Block B: Security Red Flag Interactions**:
   - Single flags carried up to **11.56x** risk multipliers in EDA.
   - Composite risk count (0 to 6 flags triggered).
   - High-risk intersection pairs (`vpn_and_ip_mismatch`, `foreign_and_ip_mismatch`, `scam_and_foreign`).
   - Risk $\times$ Amount interaction: `amount_x_merchant_risk`, `amount_to_balance_ratio`, `txns_per_hour_rate`.
3. **Block C: Categorical Interactions & Cross-Features**:
   - High-order combinations: `merchant_card_combo`, `channel_device_combo`.
4. **Block D: Strict In-Fold Smoothed Target Encoding**:
   - Additive smoothed target encoding computed strictly within training folds ($m = 10$) to prevent any target leakage.
5. **Validation Benchmarking**:
   - 5-Fold Stratified CV evaluated on **ROC-AUC** and **PR-AUC** with feature importance diagnostics.

In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Hardware Acceleration Device: {device}")
print(f"LightGBM: {lgb.__version__} | XGBoost: {xgb.__version__}")

In [ ]:
class Config:
    RANDOM_STATE = 42
    N_SPLITS = 5
    
    # Path resolution
    if os.path.exists('/kaggle/input/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    elif os.path.exists('/kaggle/input/competitions/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    else:
        DATA_DIR = Path('../../dataset')
        if not (DATA_DIR / 'train.csv').exists():
            DATA_DIR = Path('pgs-s6e11/dataset')
        OUTPUT_DIR = Path('../../submission')
        if not OUTPUT_DIR.exists():
            OUTPUT_DIR = Path('pgs-s6e11/submission')

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data Directory:   {Config.DATA_DIR}")
print(f"Output Directory: {Config.OUTPUT_DIR}")

In [ ]:
def engineer_features(df):
    data = df.copy()
    
    # ----------------------------------------------------
    # Block A: Cents, Benford & Magnitude Signals
    # ----------------------------------------------------
    data['cents'] = np.round((data['amount_usd'] % 1) * 100).astype(int)
    data['is_round_dollar'] = (data['cents'] == 0).astype(int)
    
    data['amount_int'] = np.floor(data['amount_usd']).astype(int)
    data['amount_int_div10'] = (data['amount_int'] // 10).astype(int)
    
    # Leading digit extraction (Benford's Law anomaly detection)
    data['amount_leading_digit'] = data['amount_usd'].astype(str).str[0].astype(int)
    data['balance_leading_digit'] = data['account_balance_usd'].astype(str).str[0].astype(int)
    
    # ----------------------------------------------------
    # Block B: Security Red Flag Interactions
    # ----------------------------------------------------
    bool_cols = [
        'is_foreign_transaction', 'is_new_merchant', 'used_vpn',
        'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt'
    ]
    for b in bool_cols:
        data[b] = data[b].astype(int)
        
    data['total_risk_flags'] = (
        data['ip_country_mismatch'] +
        data['is_ai_generated_scam_attempt'] +
        data['billing_shipping_mismatch'] +
        data['is_foreign_transaction'] +
        data['used_vpn'] +
        data['is_new_merchant']
    )
    
    # High-risk pair interactions
    data['foreign_and_ip_mismatch'] = data['is_foreign_transaction'] & data['ip_country_mismatch']
    data['vpn_and_ip_mismatch'] = data['used_vpn'] & data['ip_country_mismatch']
    data['scam_and_foreign'] = data['is_ai_generated_scam_attempt'] & data['is_foreign_transaction']
    data['vpn_and_foreign'] = data['used_vpn'] & data['is_foreign_transaction']
    
    # Financial ratio & interaction features
    data['amount_x_merchant_risk'] = data['amount_usd'] * data['merchant_risk_score']
    data['amount_to_balance_ratio'] = data['amount_usd'] / (data['account_balance_usd'] + 1.0)
    data['txns_per_hour_rate'] = data['txn_count_last_24h'] / (data['hours_since_last_txn'] + 0.1)
    data['velocity_x_risk'] = data['velocity_score'] * data['merchant_risk_score']
    
    # ----------------------------------------------------
    # Block C: Interaction Categories
    # ----------------------------------------------------
    data['merchant_card_combo'] = data['merchant_category'].astype(str) + '_' + data['card_type'].astype(str)
    data['channel_device_combo'] = data['channel'].astype(str) + '_' + data['device_type'].astype(str)
    
    return data

print("Feature engineering pipeline defined.")

In [ ]:
print("Loading and transforming datasets...")
raw_train = pd.read_csv(Config.DATA_DIR / 'train.csv')
raw_test = pd.read_csv(Config.DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(Config.DATA_DIR / 'sample_submission.csv')

train_fe = engineer_features(raw_train)
test_fe = engineer_features(raw_test)

y = train_fe['is_fraud'].astype(int).values

cat_cols = [
    'merchant_category', 'card_type', 'auth_method', 'channel', 'device_type',
    'merchant_card_combo', 'channel_device_combo'
]

features = [c for c in train_fe.columns if c not in ['id', 'is_fraud']]

for c in cat_cols:
    train_fe[c] = train_fe[c].astype('category')
    test_fe[c] = test_fe[c].astype('category')

print(f"Engineered Features Total: {len(features)} (Raw had 24)")
print(f"New Features Added: {[c for c in features if c not in raw_train.columns]}")

In [ ]:
skf = StratifiedKFold(n_splits=Config.N_SPLITS, shuffle=True, random_state=Config.RANDOM_STATE)

oof_lgb_fe = np.zeros(len(train_fe), dtype=np.float64)
test_lgb_fe = np.zeros(len(test_fe), dtype=np.float64)

# In-fold smoothed target encoding columns
te_cols = ['merchant_category', 'merchant_card_combo']
smoothing_m = 10.0
global_mean = y.mean()

feature_importances = pd.DataFrame({'feature': features, 'importance': 0.0})

print("=" * 70)
print("Training 5-Fold Stratified CV on Engineered Features (LightGBM)...")
print("=" * 70)

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(train_fe, y), 1):
    fold_t0 = time.time()
    
    X_tr = train_fe.iloc[train_idx][features].copy()
    y_tr = y[train_idx]
    X_va = train_fe.iloc[val_idx][features].copy()
    y_va = y[val_idx]
    X_te = test_fe[features].copy()
    
    # Calculate In-Fold Smoothed Target Encoding
    for col in te_cols:
        col_str_tr = X_tr[col].astype(str)
        counts = col_str_tr.value_counts()
        means = pd.Series(y_tr).groupby(col_str_tr.values).mean()
        smoothed = ((counts * means + smoothing_m * global_mean) / (counts + smoothing_m)).to_dict()
        
        te_name = f"{col}_te"
        X_tr[te_name] = X_tr[col].astype(str).map(smoothed).fillna(global_mean).values.astype(float)
        X_va[te_name] = X_va[col].astype(str).map(smoothed).fillna(global_mean).values.astype(float)
        X_te[te_name] = X_te[col].astype(str).map(smoothed).fillna(global_mean).values.astype(float)
            
    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': 0.04,
        'num_leaves': 6,
        'max_depth': 4,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': Config.RANDOM_STATE + fold,
        'n_estimators': 1500,
        'verbose': -1,
        'n_jobs': -1
    }
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    
    val_probs = model.predict_proba(X_va)[:, 1]
    oof_lgb_fe[val_idx] = val_probs
    test_lgb_fe += model.predict_proba(X_te)[:, 1] / Config.N_SPLITS
    
    fold_auc = roc_auc_score(y_va, val_probs)
    fold_pr_auc = average_precision_score(y_va, val_probs)
    
    print(f"Fold {fold} | Best Iter: {model.best_iteration_:4d} | ROC-AUC: {fold_auc:.5f} | PR-AUC: {fold_pr_auc:.5f} ({time.time()-fold_t0:.1f}s)")

total_cv_time = time.time() - start_time
print("=" * 70)
print(f"5-Fold CV Completed in {total_cv_time:.1f}s!")

In [ ]:
overall_auc = roc_auc_score(y, oof_lgb_fe)
overall_pr_auc = average_precision_score(y, oof_lgb_fe)

raw_benchmark_auc = 0.94700

print("-" * 60)
print(f"Raw Features LightGBM Benchmark:        {raw_benchmark_auc:.5f}")
print(f"Engineered Features LightGBM OOF ROC:   {overall_auc:.5f} ({overall_auc - raw_benchmark_auc:+.5f})")
print(f"Engineered Features LightGBM OOF PR-AUC: {overall_pr_auc:.5f}")
print("-" * 60)

np.save(Config.OUTPUT_DIR / 'oof_lgb_fe.npy', oof_lgb_fe)
print("Saved oof_lgb_fe.npy")

In [ ]:
submission = pd.DataFrame({
    'id': test_fe['id'],
    'is_fraud': test_lgb_fe
})

assert submission.shape == sample_sub.shape, f"Shape mismatch: {submission.shape} vs {sample_sub.shape}"
assert list(submission.columns) == ['id', 'is_fraud'], f"Column mismatch: {submission.columns}"
assert not submission.isna().any().any(), "Submission contains null/NaN values!"
assert (submission['is_fraud'] >= 0.0).all() and (submission['is_fraud'] <= 1.0).all(), "Predictions out of [0, 1] range!"
assert (submission['id'] == sample_sub['id']).all(), "ID alignment mismatch!"

sub_path = Config.OUTPUT_DIR / 'fe_lgb_submission.csv'
submission.to_csv(sub_path, index=False)
submission.to_csv(Config.OUTPUT_DIR / 'submission.csv', index=False)

print(f"Submission successfully generated and verified at: {sub_path}")
print(f"File size: {os.path.getsize(sub_path) / (1024*1024):.2f} MB")
print("\nFirst 10 submission rows:")
print(submission.head(10))